In [1]:
import os

In [2]:
%pwd

'c:\\Users\\param\\Desktop\\mlops_pipeline\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\param\\Desktop\\mlops_pipeline'

In [5]:
import dagshub

dagshub.init(
    repo_owner="paramchhabra",
    repo_name="mlops_pipeline",
    mlflow=True
)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

c:\Users\param\Desktop\mlops_pipeline\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=cc150bc6-96a6-422c-986b-f74730eb6cb6&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=ba683d62e54e1ed0cfc3b2d2ab965e03f2363a6a04034ce4cbcd7221013e3fd3




Accessing as paramchhabra

Initialized MLflow to track repo "paramchhabra/mlops_pipeline"

Repository paramchhabra/mlops_pipeline initialized!

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str

In [7]:
import mlProject

In [8]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [9]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifact_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name
        )

        return model_evaluation_config


In [10]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

c:\Users\param\Desktop\mlops_pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    
    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    


    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]


        dagshub.init(repo_owner="paramchhabra",repo_name="mlops_pipeline",mlflow=True)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme


        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)
            
            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

    


In [12]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-08-10 08:58:15,608:INFO:common:yaml file:config\config.yaml loaded successfully]
[2026-08-10 08:58:15,614:INFO:common:yaml file:params.yaml loaded successfully]
[2026-08-10 08:58:15,625:INFO:common:yaml file:schema.yaml loaded successfully]
[2026-08-10 08:58:15,632:INFO:common:Created directory at : artifacts]
[2026-08-10 08:58:15,635:INFO:common:Created directory at : artifacts/model_evaluation]
[2026-08-10 08:58:17,713:INFO:_client:HTTP Request: GET https://dagshub.com/api/v1/repos/paramchhabra/mlops_pipeline "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "paramchhabra/mlops_pipeline"

[2026-08-10 08:58:17,724:INFO:helpers:Initialized MLflow to track repo "paramchhabra/mlops_pipeline"]


Repository paramchhabra/mlops_pipeline initialized!

[2026-08-10 08:58:17,728:INFO:helpers:Repository paramchhabra/mlops_pipeline initialized!]
[2026-08-10 08:58:20,161:INFO:common:json file saved at :artifacts\model_evaluation\metrics.json]


2026/08/10 08:58:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/10 08:58:21 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\param\Desktop\mlops_pipeline
2026/08/10 08:58:32 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\param\Desktop\mlops_pipeline
2026/08/10 08:58:32 INFO mlflow.utils.environment: Detected uv project at c:\Users\param\Desktop\mlops_pipeline. Attempting to export requirements via 'uv export'.
2026/08/10 08:58:32 INFO mlflow.utils.uv_utils: Exported 196 dependencies via uv
2026/08/10 08:58:32 INFO mlflow.utils.environment: Successfully exported 196 requirements from uv project. Skipping package capture based inference.
2026/08/10 08:58:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'Elast

🏃 View run efficient-finch-706 at: https://dagshub.com/paramchhabra/mlops_pipeline.mlflow/#/experiments/0/runs/3c67361cdbe44b268600f6721dfaa7b9
🧪 View experiment at: https://dagshub.com/paramchhabra/mlops_pipeline.mlflow/#/experiments/0
